## Проверка работы LLM с разными промптами

In [ ]:
import os
import random
import sys
import time
import yaml
from typing import Union
from pathlib import Path

from langchain_core.messages import BaseMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, HarmBlockThreshold, HarmCategory

sys.path.append("..")

import src.llm.prompts as prompts
from src.app_config import TEMPERATURE
from src.llm.prompts import text_question_answer_template
from src.llm.llm_manager import LoggerChatModel



def load_yaml_file(yaml_path: Path) -> dict:
    """Загрузить настройки из YAML файла конфигурации"""
    try:
        with open(yaml_path, "r", encoding="UTF-8") as stream:
            return yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        raise yaml.YAMLError(f"Ошибка в чтении файла {yaml_path}: {exc}")


class GeminiModel():
    """Получить доступ к модели OpenAI"""

    def __init__(self, api_key: str, llm_model: str, llm_proxy: Union[str, None] = None) -> None:
        self.llm_proxy = llm_proxy
        self.model = llm_model
        self.google_api_key= api_key

    def invoke(self, prompt: ChatPromptTemplate) -> BaseMessage:
        prompt_messages = [SystemMessage(content=prompts.custom_instructions)] + prompt.messages
        # prompt_messages = prompt.messages
        # случайно выбираем одну прокси за другой, пока запрос к LLM не пройдет
        llm_proxies = self.llm_proxy.copy()
        random.shuffle(llm_proxies)

        for proxy in llm_proxies:
            os.environ["https_proxy"] = proxy
            model = ChatGoogleGenerativeAI(
                model=self.model,
                google_api_key=self.google_api_key,
                temperature=TEMPERATURE,
                safety_settings={
                    HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_DEROGATORY: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_TOXICITY: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_VIOLENCE: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_SEXUAL: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_MEDICAL: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_DANGEROUS: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                
                },
                # thinking_budget = 0,
            )
            try:
                response = model.invoke(prompt_messages)
                return response
            except Exception:
                time.sleep(3)

llm_api_key = load_yaml_file("../data_folder/secrets/secrets.yaml")["llm_api_key"]
llm_proxy = load_yaml_file("../data_folder/secrets/secrets.yaml")["llm_proxy"]
# llm_model = "gemini-2.5-flash-preview-04-17"
llm_model = "gemini-2.0-flash"


resume = {'general_knowledge_questions': '', 
          'personal_information': {
              'first_name': 'Аристаний', 
              'last_name': 'Звяегольцев', 
              'middle_name': 'Астромерович', 
              'current_city': 'Москва', 
              'metro': '', 
              'has_vehicle': False, 
              'birthday': '', 
              'age': '', 'sex': 
              'Мужской', 
              'citizenship': ['Россия'], 
              'legal_authorization': 'Россия', 
              'linkedIn': 'https://linkedin.com/in/alex-semenov-f3e57c712', 
              'skype': 'aristaniy93', 
              'other_site': 'https://www.aristaniy93.ru', 
              'moi_krug': 'https://career.habr.ru/alexneth93', 
              'livejournal': 'https://www.livejournal.com/alexneth93', 
              'telegram': 'https://t.me/aristaniy93', 
              'phone': '+7 (933) 575-35-35', 
              'email': 'aristaniy93@gmail.com', 
              'preferred_contact': 'email'}, 
              'work_preferences': {
                  'position': 'Программист Python', 
                  'can_relocate': 'не могу переехать', 
                  'professional_roles': ['Аналитик', 'Программист, разработчик'], 
                  'employments': ['Полная занятость'], 
                  'schedules': ['Полный день', 'Удаленная работа'], 
                  'travel_time_to_work': 'Не более часа', 
                  'ready_to_business_trips': 'не готов к командировкам'}, 
                'availability': {'notice_period': '2 недели'}, 
                'languages': {'Русский': 'Родной'}, 
                'education_details': {
                    'level': 'Высшее', 
                    'primary': [
                        {'name': 'Московский государственный университет имени М.В. Ломоносова, Москва', 'organization': 'ВМК', 'year': 2019, 'university_acronym': 'МГУ', 'education_level': 'Высшее'}], 
                    'elementary': [
                        {'name': 'Московский государственный технический университет имени Н.Э. Баумана (национальный исследовательский университет), Москва', 'year': 2024}], 
                    'additional': [
                        {'name': 'Python разработчик', 'organization': 'Яндекс Практикум', 'result': 'Python разработчик', 'year': 2021}], 
                    'attestation': [
                        {'name': 'PCPP1™ – Certified Professional Python Programmer Level 1', 'organization': 'Python Institute', 'result': 'Python Programmer', 'year': 2024}]}, 
                    'certifications': [
                        {'type': 'custom', 'title': 'Certified Associate in Python Programming', 'achieved_at': '2024-01-01', 'url': 'https://www.pluralsight.com/cloud-guru/courses/certified-associate-in-python-programming-certification-pcap-31-03?clickid=EAIaIQobChMI482T9IaWigMVCpCDBx0HrgzdEAAYAyAAEgIS6fD_BwE&utm_source=google&utm_medium=paid-search&utm_campaign=upskilling-and-reskilling&utm_term=ssi-emea-dynamic&utm_content=free-trial&gad_source=1&gclid=EAIaIQobChMI482T9IaWigMVCpCDBx0HrgzdEAAYAyAAEgIS6fD_BwE'}, 
                        {'type': 'custom', 'title': 'Django Certified Solutions Architect', 'achieved_at': '2023-01-01', 'url': 'https://www.tealhq.com/certifications/python-django-developer'}], 
                    'recommendation': [
                        {'name': 'Михаил', 'organization': 'ПРАЙМ ГРУП', 'position': 'Генеральный директор', 'contact': ''}], 
                    'experience_details': {
                        'total_experience_years': 4, 
                        'details': [
                            {'start_date': '2022-08-01', 'company': 'ООО «Green-Park»', 'industries': [], 'position': 'Python разработчик', 'description': 'Обязанности:\n- разработка веб-платформы для промо-кампании\n- разработка телеграм-бота для промо-кампании\n\nДостижения:\n      - Организовал структуру проекта\n      - Cоздал посадочную страницу на React, обеспечивающую UX/UI-оптимизацию и взаимодействие с пользователем\n      - Внедрил асинхронную обработку чеков API Федеральной налоговой службы\n      - Интегрировал API Яндекс Карт\n      - Ускорил деплой на 10 минут благодаря Git + Docker-compose\n      - Настроил веб-сервер при помощи Nginx и Сertbot'}, 
                            {'start_date': '2020-09-01', 'end_date': '2022-06-01', 'company': 'ПРАЙМ ГРУП', 'industries': [], 'position': 'Python разработчик', 'description': 'Обязанности:\n- разработка телеграм-бота на Aiogram для взаимодействия с клиентами\n- создание сайта-сборника проектов на Django\n\nДостижения:\n      - Разработал базу данных для хранения и обновления рабочего расписания\n      - Внедрил систему мгновенных уведомлений, позволяющую оповещать клиентов об изменениях в заказах в режиме реального времени\n      - Интегрировал API Яндекс Карт и Яндекс Погоды'}]}, 
                    'salary_expectations': {'amount': 300000, 'currency': 'RUR'}, 
                    'skills': ['Python', 'SQL', 'Linux', 'PostgreSQL', 'Git', 'Django Framework', 'Английский язык', 'React', 'Redis', 'JavaScript', 'Docker', 'FastAPI', 'Aiogram', 'REST', 'HTML', 'Clickhouse', 'CSS', 'Celery', 'RabbitMQ', 'Unit Testing', 'Apache Airflow', 'Flask'], 
                    'about_me': 'Меня зовут Аристаний и я специализируюсь в области web-программирования на Python.\n\nЛичные достижения:\n- Победитель хакатона - Занял первое место в хакатоне IT Inno Hack 2023\n- Создатель популярного проекта mqtt-packet-parser (собрал более 300 звезд на GitHub)\n\nВ список моих интересов входят:\n- Чат-боты\n  - Машинное обучение и искусственный интеллект\n  - Computer Vision/CV/Компьютерное зрение\n  - Natural language processing/NLP/Обработка естественного языка\n  - Кибербезопасность\n  - Antifraud/Выявление мошеннических действий\n\nМои проекты:\n- https://github.com/aristaniy93/client_int_bot.git\nClient Interaction Telegram Bot - Телеграм-бот на Aiogram для взаимодействия с клиентами\n\n- https://github.com/aristaniy93/mqtt_packet_parser.git\nМодуль Node.js для анализа пакетов MQTT, эффективность анализа повышена на 40%\n'}
current_date = "08.04.2025"
sex = "male"
question = "Работал ли с FPGA?"

prompt_template = ChatPromptTemplate.from_template(text_question_answer_template)

llm = GeminiModel(llm_api_key, llm_model, llm_proxy)
llm_cheap = LoggerChatModel(llm)

chain = prompt_template | llm_cheap

result = chain.invoke({
    "resume": resume,
    "current_date": current_date,
    "sex": sex,
    "question": question
    })

# Print the result
print(result.content)

2025-05-05 14:50:20.242 | INFO     | src.llm.llm_manager:__init__:359 - LoggerChatModel успешно инициализирован, LLM: <__main__.GeminiModel object at 0x73eb76ca2b10>
2025-05-05 14:50:20.244 | INFO     | src.llm.llm_manager:__call__:368 - Попытка вызова LLM
2025-05-05 14:50:21.251 | INFO     | src.llm.llm_manager:parse_llmresult:412 - Парсинг результата LLM
2025-05-05 14:50:21.253 | INFO     | src.llm.llm_manager:__call__:374 - Успешно распарсили результат работы LLM: {'content': 'Нет информации.', 'response_metadata': {'model_name': 'gemini-2.0-flash', 'system_fingerprint': '', 'finish_reason': 'STOP', 'logprobs': None}, 'id': 'run-38555b90-2c47-49f1-9260-f216802767f2-0', 'usage_metadata': {'input_tokens': 2164, 'output_tokens': 4, 'total_tokens': 2168}}
2025-05-05 14:50:21.254 | INFO     | src.llm.llm_manager:log_request:233 - Начинается выполнение метода log_request
2025-05-05 14:50:21.256 | INFO     | src.llm.llm_manager:log_request:234 - Получены промпты
2025-05-05 14:50:21.257 | I